# Advance Crime Data Pipeline

## Stage 3: Feature Engineering and Transformation

**Stakeholder:** Police Force Analytics Unit <br>
**Stage:** 3 of 4: Feature Engineering and Enrichment <br>
**Medallion Layer:** Silver 🥈<br>
**Police Forces:** West Midlands · Thames Valley · Surrey · Cumbria <br>
**Authors:** Group 1 <br>
**Last Updated:** 21 May 2026

---

### Purpose

This notebook enriches the cleaned Silver crime data by joining three external datasets at the appropriate grain. The output is a fully enriched dataset ready for aggregation in the Gold layer, enabling socioeconomic comparisons across regions.

---

### Enrichment Datasets

| Dataset | Source | Grain | Join Key | Analytical Value |
|---|---|---|---|---|
| Population | ONS Mid-2024 district estimates | District | `district_name` | Enables crime rate normalisation per 1,000 residents |
| House Prices | Land Registry 2024 median prices | District | `district_name` | Provides socioeconomic context at district level |
| Deprivation (IMD) | MHCLG IMD 2024 rankings | District | `district_name` | Provides deprivation context across multiple dimensions |
| Coordinates | District centroid coordinates | District | `district_name` | Enables geographic mapping in Power BI |
---

### Join Strategy & Assumptions

| Dataset | Join Type | Assumption / Limitation |
|---|---|---|
| Population | LEFT JOIN on `lsoa_code` | Records with suppressed locations (`NOT_RECORDED`) will not match — null population is expected |
| House Prices | LEFT JOIN on `lsoa_code` + `month_num` | Crime data is from 2026 but house prices only available to 2023. **2023 is used as a proxy** — most recent available year. Year is excluded from the join key |
| Deprivation | LEFT JOIN on `force_name` | IMD is England-only — Dyfed-Powys (Wales) will not match and will have null deprivation values. This is a known limitation |


## 1. Environment Setup

This section sets up the Snowflake and Python environment required for the pipeline. It creates the necessary **warehouse**, **database**, and **schemas** if they do not already exist.

In [ ]:
%%sql -r dataframe_1
USE ROLE SYSADMIN;

-- Create a warehouse for compute
CREATE WAREHOUSE IF NOT EXISTS CRIME_WH
  WAREHOUSE_SIZE = 'X-SMALL'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE;

-- Create the database
CREATE DATABASE IF NOT EXISTS CRIME_PIPELINE;

-- Create schemas for each pipeline stage
CREATE SCHEMA IF NOT EXISTS CRIME_PIPELINE.RAW;        -- Bronze
CREATE SCHEMA IF NOT EXISTS CRIME_PIPELINE.CLEAN;      -- Silver
CREATE SCHEMA IF NOT EXISTS CRIME_PIPELINE.REPORTING;  -- Gold

In [ ]:
import pandas as pd
from snowflake.snowpark.context import get_active_session
from snowflake.connector.pandas_tools import write_pandas

# Establish active Snowpark session
session = get_active_session()
session.sql("USE DATABASE CRIME_PIPELINE").collect()
session.sql("USE SCHEMA CLEAN").collect()

# Table references
SILVER_TABLE   = "CRIME_PIPELINE.CLEAN.SILVER_CRIME_CLEAN"
ENRICHED_TABLE = "SILVER_CRIME_ENRICHED"

# Enrichment stage and file references
STAGE    = "@CRIME_PIPELINE.RAW.ENRICHMENT_STAGE"
POP_FILE = "population_2024_clean.csv"
HP_FILE  = "house_prices_2024_clean.csv"
DEP_FILE = "iod_clean.csv"
COORD_FILE = "coordinates_clean.csv"


# House price proxy year -- most recent available year in the dataset
# Crime data is from 2026 but house prices only available to 2023
HP_PROXY_YEAR = 2023

print("Session database :", session.get_current_database())
print("Session schema   :", session.get_current_schema())

## 2. Read from Silver Table and Initial Inspection

Load the cleaned crime dataset from `SILVER_CRIME_CLEAN`. A `district_name` column is derived from `lsoa_name` and a `force_name` column is derived from `falls_within` — both are used as join keys for the enrichment datasets.

In [ ]:
# Read cleaned crime data from Silver table
crime = session.table(SILVER_TABLE).to_pandas()

# Standardise column names: strip whitespace, lowercase, replace spaces with underscores
crime.columns = [c.strip().lower().replace(" ", "_") for c in crime.columns]

# Derive district_name from lsoa_name by splitting on the last space
# e.g. 'Mole Valley 012A' -> 'Mole Valley'
crime["district_name"] = (
    crime["lsoa_name"]
    .fillna("NOT_RECORDED")
    .str.rsplit(" ", n=1)
    .str[0]
)

# Derive force_name from falls_within by stripping ' Police' and ' Constabulary' suffixes
# e.g. 'West Midlands Police' -> 'West Midlands', 'Cumbria Constabulary' -> 'Cumbria'
crime["force_name"] = (
    crime["falls_within"]
    .str.replace(" Police", "", regex=False)
    .str.replace(" Constabulary", "", regex=False)
    .str.strip()
)

# Ensure year and month_num are numeric for downstream use
crime["year"]      = pd.to_numeric(crime["year"],      errors="coerce").astype("Int64")
crime["month_num"] = pd.to_numeric(crime["month_num"], errors="coerce").astype("Int64")

# Record baseline row count for reconciliation report
baseline_count = len(crime)

print("Silver rows read :", len(crime))
print("Force names      :", crime["force_name"].unique().tolist())
print("District sample  :", crime["district_name"].unique()[:5].tolist())

In [ ]:
# Visual inspection of the first five rows to confirm structure and raw values
crime.head()

## 3. Load Population Data

ONS Mid-2024 population estimates at district level. Used to calculate normalised crime rates (crimes per 1,000 residents) in the Gold layer.

**Grain:** One row per district <br>
**Join key:** `district_name` <br>
**Columns:** `district_name`, `population_2024`

In [ ]:
pop = pd.read_csv(
    session.file.get_stream(f"{STAGE}/{POP_FILE}")
)

# Standardise column names
pop.columns = [c.strip().lower().replace(" ", "_") for c in pop.columns]

# Validate -- duplicate districts would cause row fan-out on join
dup = pop.duplicated(subset=["district_name"]).sum()
print(f"Population rows  : {len(pop):,}")
print(f"Duplicate distrs : {dup}")
print(f"Null population  : {pop['population_2024'].isnull().sum()}")
print(pop.head(3))

## 4. Load House Price Data

Land Registry 2024 median house prices at district level.

**Grain:** One row per district <br>
**Join key:** `district_name` <br>
**Columns:** `district_name`, `median_house_price_2024`

In [ ]:
hp = pd.read_csv(
    session.file.get_stream(f"{STAGE}/{HP_FILE}")
)

# Standardise column names
hp.columns = [c.strip().lower().replace(" ", "_") for c in hp.columns]

# Validate -- no duplicates at join grain
dup = hp.duplicated(subset=["district_name"]).sum()
print(f"House price rows : {len(hp):,}")
print(f"Duplicate distrs : {dup}")
print(hp.head(3))

## 5. Load Deprivation Data

MHCLG Index of Multiple Deprivation (IMD) 2024 rankings at district level.

**Grain:** One row per district <br>
**Join key:** `district_name` <br>
**Limitation:** IMD covers England only — Welsh districts will have null deprivation values after the join.

In [ ]:
dep = pd.read_csv(
    session.file.get_stream(f"{STAGE}/{DEP_FILE}"),
    dtype=str
)

# Standardise column names
dep.columns = [c.strip().lower().replace(" ", "_") for c in dep.columns]

# Rename columns to clean, consistent names matching the target output
dep = dep.rename(columns={
    "local_authority_district_name_(2024)"        : "district_name",
    "index_of_multiple_deprivation_(imd)_rank_2024": "imd_rank_2024",
    "income_rank_2024"                             : "income_rank_2024",
    "employment_rank_2024"                         : "employment_rank_2024",
    "health_deprivation_and_disability_rank_2024"  : "health_rank_2024",
    "crime_rank_2024"                              : "crime_rank_2024"
})

# Retain only required columns
dep = dep[["district_name", "imd_rank_2024", "income_rank_2024",
           "employment_rank_2024", "health_rank_2024", "crime_rank_2024"]]

# Cast rank columns to numeric
rank_cols = [c for c in dep.columns if "rank" in c]
dep[rank_cols] = dep[rank_cols].apply(pd.to_numeric, errors="coerce")

# Validate
dup = dep.duplicated(subset=["district_name"]).sum()
print(f"Deprivation rows : {len(dep):,}")
print(f"Duplicate distrs : {dup}")
print(dep.head(3))

## 6. Load Coordinate Data

District centroid coordinates for geographic mapping in Power BI.

**Grain:** One row per district <br>
**Join key:** `district_name` <br>
**Columns:** `district_name`, `latitude`, `longitude`

In [ ]:
coords = pd.read_csv(
    session.file.get_stream(f"{STAGE}/{COORD_FILE}")
)

# Standardise column names
coords.columns = [c.strip().lower().replace(" ", "_") for c in coords.columns]

# Validate
dup = coords.duplicated(subset=["district_name"]).sum()
print(f"Coordinate rows  : {len(coords):,}")
print(f"Duplicate distrs : {dup}")
print(coords.head(3))

## 7. Join All Enrichment Datasets

In this step, four sequential LEFT JOINs are applied to the crime dataset on `district_name`. All crime records are retained regardless of whether an enrichment match exists. A row count assertion after each join confirms no fan-out has occurred.

In [ ]:
# Join 1: Population on district_name
crime_enriched = crime.merge(pop, on="district_name", how="left")

# Validation check
assert len(crime_enriched) == baseline_count, \
    f"Row count mismatch after population join: {len(crime_enriched)} vs {baseline_count}"


print(f"After population join: {len(crime_enriched):,} rows")
print(f"Null population_2024: {crime_enriched['population_2024'].isnull().sum():,}")

In [ ]:
# Join 2: House Prices on district_name
crime_enriched = crime_enriched.merge(hp, on="district_name", how="left")

assert len(crime_enriched) == baseline_count, \
    f"Row count mismatch after house price join: {len(crime_enriched)} vs {baseline_count}"

print(f"After house price join              : {len(crime_enriched):,} rows")
print(f"Null median_house_price_2024        : {crime_enriched['median_house_price_2024'].isnull().sum():,}")

In [ ]:
# Join 3: Deprivation on district_name
crime_enriched = crime_enriched.merge(dep, on="district_name", how="left")

assert len(crime_enriched) == baseline_count, \
    f"Row count mismatch after deprivation join: {len(crime_enriched)} vs {baseline_count}"

print(f"After deprivation join : {len(crime_enriched):,} rows")
print(f"Null imd_rank_2024     : {crime_enriched['imd_rank_2024'].isnull().sum():,}")

In [ ]:
print(crime_enriched.columns.tolist())

In [ ]:
# ── Join 4: Coordinates on district_name ─────────────────────────────────────
crime_enriched = crime_enriched.merge(coords, on="district_name", how="left")

assert len(crime_enriched) == baseline_count, \
    f"Row count mismatch after coordinates join: {len(crime_enriched)} vs {baseline_count}"

print(f"After coordinates join : {len(crime_enriched):,} rows")
print(f"Null latitude          : {crime_enriched['latitude'].isnull().sum():,}")

## 8. Post-Join Reconciliation Report

Document match rates for all four enrichment datasets. Null values are expected for specific known cases — these are documented limitations, not data quality issues.

In [ ]:
print("╔══════════════════════════════════════════════════════════════╗")
print("║           ENRICHMENT RECONCILIATION REPORT                  ║")
print("╚══════════════════════════════════════════════════════════════╝")
print(f"Baseline rows (Silver)       : {baseline_count:,}")
print(f"Rows after all joins         : {len(crime_enriched):,}")

print("\n── Match rates ──")
print(f"Population matched    : {crime_enriched['population_2024'].notnull().sum():,} "
      f"({crime_enriched['population_2024'].notnull().mean()*100:.1f}%)")
print(f"House price matched   : {crime_enriched['median_house_price_2024'].notnull().sum():,} "
      f"({crime_enriched['median_house_price_2024'].notnull().mean()*100:.1f}%)")
print(f"Deprivation matched   : {crime_enriched['imd_rank_2024'].notnull().sum():,} "
      f"({crime_enriched['imd_rank_2024'].notnull().mean()*100:.1f}%)")
print(f"Coordinates matched   : {crime_enriched['latitude'].notnull().sum():,} "
      f"({crime_enriched['latitude'].notnull().mean()*100:.1f}%)")

print("\n── Known null causes ──")
print("Deprivation nulls  : IMD is England-only — Welsh districts not covered")
print("All other nulls    : suppressed location records (NOT_RECORDED district)")

## 9. Inspect Enriched Dataset

In [ ]:
print("Shape  :", crime_enriched.shape)
print("Columns:", crime_enriched.columns.tolist())

crime_enriched.head()

## 10. Export to Enriched Silver Table

Persist the fully enriched dataset to `CRIME_PIPELINE.CLEAN.SILVER_CRIME_ENRICHED`. This is the final hand-off before Gold aggregation.

In [ ]:
# Reset index before writing to suppress non-standard index warning
crime_enriched = crime_enriched.reset_index(drop=True)

success, nchunks, nrows, _ = write_pandas(
    conn=session.connection,
    df=crime_enriched,
    table_name=ENRICHED_TABLE,
    database="CRIME_PIPELINE",
    schema="CLEAN",
    auto_create_table=True,  # creates table if it does not exist
    overwrite=True           # replaces existing data on each run
)

# Verify written row count matches enriched row count
written_count = session.table(f"CRIME_PIPELINE.CLEAN.{ENRICHED_TABLE}").count()
assert written_count == len(crime_enriched), \
    f"Row count mismatch: {written_count} written vs {len(crime_enriched)} expected"

print(f"Enriched Silver table written successfully")
print(f"Table  : CRIME_PIPELINE.CLEAN.{ENRICHED_TABLE}")
print(f"Rows   : {written_count:,}")